In [145]:
from dbfread import DBF
import pandas as pd
import numpy as np

## Data preparation
### Raw data ingestion

In [146]:
# read the raw data and store in a dataframe
dbf = DBF('data/raw/exped.DBF')
df = pd.DataFrame(iter(dbf))

In [147]:
# identify empty strings as missing values
df.replace("", np.nan, inplace=True)

# ensure there are now empty rows or columns
df.dropna(how='all', axis=0, inplace=True)
df.dropna(how='all', axis=1, inplace=True)

# remove any duplicates
df.drop_duplicates(inplace=True)

In [148]:
# standardize column names
df.columns = map(lambda x: x.lower(), df.columns)

In [149]:
df.shape

(11578, 66)

In [150]:
df.head()

,expid,peakid,year,season,host,route1,route2,route3,route4,nation,...,accidents,achievment,agency,comrte,stdrte,primrte,primmem,primref,primid,chksum
0,ANN260101,ANN2,1960,1,1,NW Ridge-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2442047
1,ANN269301,ANN2,1969,3,1,NW Ridge-W Ridge,NaN,NaN,NaN,Yugoslavia,...,Draslar frostbitten hands and feet,NaN,NaN,None,None,False,False,None,NaN,2445501
2,ANN273101,ANN2,1973,1,1,W Ridge-N Face,NaN,NaN,NaN,Japan,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2446797
3,ANN278301,ANN2,1978,3,1,N Face-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2448822
4,ANN279301,ANN2,1979,3,1,N Face-W Ridge,NW Ridge of A-IV,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2449204


### Data Cleaning

In [151]:
df.groupby('expid').expid.count().sort_values(ascending=False)[:1]

expid
KANG10101    2
Name: expid, dtype: int64

In [152]:
df.loc[df.expid == 'KANG10101']

,expid,peakid,year,season,host,route1,route2,route3,route4,nation,...,accidents,achievment,agency,comrte,stdrte,primrte,primmem,primref,primid,chksum
2860,KANG10101,KANG,1910,1,3,NE Side (recon),NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2405
6780,KANG10101,KANG,2010,1,1,SW Face,NaN,NaN,NaN,S Korea,...,NaN,NaN,Windhorse Trekking,False,True,False,False,False,NaN,2457821


> Some _expid_ values are duplicated for expeditions to the same peak occuring one century appart

In [153]:
# add the full year to the expedition id to ensure uniqueness
df.expid = df.expid.str.cat(df.year)
assert df.expid.nunique() == df.shape[0]

In [154]:
# map country index to name as per the documentation
host_map = {
	0: 'Unknown',
	1: 'Nepal',
	2: 'China',
	3: 'India'
}

df.host = df.host.map(host_map)

In [155]:
# map season index to name as per the documentation
season_map = {
	0: 'Unknown',
	1: 'Spring',
	2: 'Summer',
	3: 'Autumn',
	4: 'Winter'
}

df.season = df.season.map(season_map)

In [156]:
# remove expedition with undefined main route
df = df.loc[df.route1.notna()]

In [157]:
# remove expeditions that include non-climbing activities
df = df.loc[~df.traverse & ~df.ski & ~df.parapente]

df.drop(['traverse', 'ski', 'parapente'], axis=1, inplace=True)

In [158]:
# filter based on expedition termination reason:
# 12 - Did not attempt climb
# 13 - Attempt rumored
df = df.loc[
	~df.termreason.isin([12, 13])
]

df.drop('termreason', axis=1, inplace=True)

In [159]:
# reason_map = {
# 	0: "Unknown",
# 	1: "Success (main peak)",
# 	2: "Success (subpeak, foresummit)",
# 	3: "Success (claimed)",
# 	4: "Bad weather (storms, high winds)",
# 	5: "Bad conditions (deep snow, avalanching, falling ice, or rock)",
# 	6: "Accident (death or serious injury)",
# 	7: "Illness, AMS, exhaustion, or frostbite",
# 	8: "Lack (or loss) of supplies, support or equipment",
# 	9: "Lack of time",
# 	10: "Route technically too difficult, lack of experience, strength, or motivation",
# 	11: "Did not reach base camp",
# 	12: "Did not attempt climb",
# 	13: "Attempt rumored",
# 	14: 'Other'
# }
#
# df.termreason = df.termreason.map(reason_map)

In [160]:
# remove unused columns
df.drop([
	'route2', 'route3', 'route4', 'success2', 'success3', 'success4', 'ascent2', 'ascent3', 'ascent4', 'claimed',
	'disputed', 'approach', 'smtdate', 'smttime', 'smtdays', 'totdays', 'termdate', 'termnote', 'highpoint',
	'smtmembers', 'mdeaths', 'smthired', 'hdeaths', 'othersmts', 'campsites', 'routememo', 'accidents', 'achievment',
	'primmem', 'primref', 'primid', 'chksum', 'leaders', 'countries', 'ascent1', 'bcdate', 'o2used', 'o2none',
	'o2medical', 'o2unkwn', 'agency', 'o2taken', 'nohired', 'rope'], axis=1, inplace=True)

In [161]:
df.shape

(10888, 18)

> After initial cleaning and filtering, we are left with a dataset of 11,400 expeditions

In [162]:
df['sponsored'] = df.sponsor.notna()
df.drop('sponsor', axis=1, inplace=True)

In [163]:
df.head()

,expid,peakid,year,season,host,route1,nation,success1,camps,totmembers,tothired,o2climb,o2descent,o2sleep,comrte,stdrte,primrte,sponsored
0,ANN2601011960,ANN2,1960,Spring,Nepal,NW Ridge-W Ridge,UK,True,6,10,9,True,False,True,None,None,False,False
1,ANN2693011969,ANN2,1969,Autumn,Nepal,NW Ridge-W Ridge,Yugoslavia,True,6,10,0,False,False,False,None,None,False,True
2,ANN2731011973,ANN2,1973,Spring,Nepal,W Ridge-N Face,Japan,True,5,6,8,False,False,False,None,None,False,True
3,ANN2783011978,ANN2,1978,Autumn,Nepal,N Face-W Ridge,UK,False,0,2,0,False,False,False,None,None,False,True
4,ANN2793011979,ANN2,1979,Autumn,Nepal,N Face-W Ridge,UK,False,0,3,0,False,False,False,None,None,False,False


In [164]:
df.shape

(10888, 18)